# Error by Geography - 500 Epoch Linear Scaling (Stability Analysis)

This notebook analyses whether geographic patterns of reconstruction error are stable
across retraining runs using the **500-epoch linear scaling** models
(`retraining_stability_500epochs_linscaling`).

**Approach:**
- Load reconstruction errors from all retraining runs (500 epochs, linear scaling)
- Compute **mean** error across runs for each OA
- Plot AE (mean ± std across runs) vs PCA by geographic grouping
- Both `bar` and `point_only` (dot) diff styles for the bottom panels
- Spaghetti plots and rank stability analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from textwrap import fill
from scipy.stats import spearmanr

# Set style
sns.set(style="white")

# Configuration
bottleneck = 100  # Focus on 100D

print(f"Analyzing reconstruction error stability by geography (500 epochs, linear scaling)")
print(f"Bottleneck dimension: {bottleneck}")

## 1. Load Auxiliary Geographic Data

In [ ]:
# Load the lookup tables
oa_lsoa = pd.read_csv('../data/geofiles/lookup_oa2022_lsoa11_EW.csv')
oa_lsoa.set_index('OA21CD', inplace=True)

oa_msoa = pd.read_csv('../data/geofiles/Output_Area_to_Lower_layer_Super_Output_Area_to_Middle_layer_Super_Output_Area_to_Local_Authority_District_(December_2021)_Lookup_in_England_and_Wales_v3.csv')[["OA21CD","MSOA21CD"]].set_index('OA21CD')

# Load the IMD data
imd = pd.read_csv('../data/geofiles/uk_imd2019.csv')
imd = imd[["LSOA","SOA_decile"]]
imd.columns = ['LSOA11CD','IMD']

# Load the population density data
density = pd.read_csv('../data/census_data/eng_raw_csvs/ts006.csv')
density.columns = ['OA21CD','Density']
density['Density_decile'] = pd.qcut(density['Density'], 10, labels=False)
density['Density_decile'] = 10 - density['Density_decile']  # reverse the order
density.drop('Density', axis=1, inplace=True)
density.set_index('OA21CD', inplace=True)

print(f"Loaded OA-LSOA lookup: {len(oa_lsoa)} rows")
print(f"Loaded OA-MSOA lookup: {len(oa_msoa)} rows")
print(f"Loaded IMD data: {len(imd)} rows")
print(f"Loaded density data: {len(density)} rows")

In [ ]:
# Load OAC data
OAC = pd.read_csv("../data/OAC/OAC_assignment.csv")
OAC = OAC[["Geography_Code", "Supergroup8", "Group", "Subgroup"]]
OAC = OAC.rename(columns={"Geography_Code": "OA21CD"})

# Load the OAC category names
OAC_cats = pd.read_csv("../data/OAC/OAC_cats.csv")
OAC_cats = OAC_cats[['Classification Code', 'Classification Name']]
OAC_cats_dict = OAC_cats.set_index('Classification Code')['Classification Name'].to_dict()

OAC['Supergroup8'] = OAC['Supergroup8'].astype(str)
OAC['Supergroup_name'] = OAC['Supergroup8'].map(OAC_cats_dict)
OAC['Supergroup_codename'] = OAC['Supergroup8'] + " - " + OAC['Supergroup_name']
OAC['Group'] = OAC['Group'].astype(str)
OAC['Group_name'] = OAC['Group'].map(OAC_cats_dict)
OAC['Subgroup'] = OAC['Subgroup'].astype(str)
OAC['Subgroup_name'] = OAC['Subgroup'].map(OAC_cats_dict)

print(f"Loaded OAC data: {len(OAC)} rows")
print(f"Unique supergroups: {OAC['Supergroup_codename'].nunique()}")

## 2. Load Reconstruction Errors from Stability Runs

In [ ]:
# Load the stability results (500 epochs, linear scaling)
stability_path = f"../AE_outputs/retraining_stability_500epochs_linscaling/data/stability_checkpoint_{bottleneck}d.pkl"

with open(stability_path, 'rb') as f:
    checkpoint = pickle.load(f)

n_runs = checkpoint['n_runs']

print(f"Loaded checkpoint for {bottleneck}D (500 epochs, linear scaling)")
print(f"Keys: {checkpoint.keys()}")
print(f"Number of runs: {n_runs}")

reco_errors_list = checkpoint['reco_errors_list']
print(f"Shape of reco errors for run 0: {reco_errors_list[0].shape}")

In [ ]:
# Load census data to get OA identifiers
cleaned_data_path = "../data/census_data/engcensus_cleaned_scaled.parquet"
data = pd.read_parquet(cleaned_data_path)
data = data.reset_index()

oa_ids = data['OA'].values
print(f"Number of OAs: {len(oa_ids)}")
assert len(oa_ids) == len(reco_errors_list[0]), "OA count mismatch!"

In [ ]:
# Load PCA reconstruction for comparison
pca_path = f"../AE_outputs/engcensus_all/PCA/{bottleneck}_components.csv"
pca_reco = pd.read_csv(pca_path, index_col=0)

pca_err = np.sqrt(np.mean((data.set_index("OA") - pca_reco) ** 2, axis=1)).reset_index()
pca_err.columns = ['OA21CD', 'pca_err']
pca_err['pca_err'] = pca_err['pca_err'] * 100

print(f"PCA reconstruction error computed")
print(f"Mean PCA RMSE: {pca_err['pca_err'].mean():.4f}%")

In [ ]:
# Create combined DataFrame with all runs
reco_err_dfs = []
for run_idx, reco_err in enumerate(reco_errors_list):
    rmse = np.sqrt(reco_err) * 100
    df_run = pd.DataFrame({'OA21CD': oa_ids, f'reco_err_run_{run_idx}': rmse})
    reco_err_dfs.append(df_run)

reco_err_all = reco_err_dfs[0]
for df in reco_err_dfs[1:]:
    reco_err_all = reco_err_all.merge(df, on='OA21CD')

run_cols = [f'reco_err_run_{i}' for i in range(n_runs)]
reco_err_all['ae_mean'] = reco_err_all[run_cols].mean(axis=1)
reco_err_all['ae_std'] = reco_err_all[run_cols].std(axis=1)
reco_err_all['ae_cv'] = reco_err_all['ae_std'] / reco_err_all['ae_mean'] * 100

print(f"Combined DataFrame: {reco_err_all.shape}")
print(f"Mean AE RMSE across runs: {reco_err_all['ae_mean'].mean():.4f}%")
print(f"Mean Std across OAs: {reco_err_all['ae_std'].mean():.4f}%")
print(f"Mean CV across OAs: {reco_err_all['ae_cv'].mean():.2f}%")

In [ ]:
# Merge with geographic metadata
reco_err_all = reco_err_all.merge(OAC, on="OA21CD", how="left")
reco_err_all = reco_err_all.merge(density, on="OA21CD", how="left")
reco_err_all = reco_err_all.merge(oa_lsoa, on="OA21CD", how="left")
reco_err_all = reco_err_all.merge(oa_msoa, on="OA21CD", how="left")
reco_err_all = reco_err_all.merge(imd, on="LSOA11CD", how="left").dropna()
reco_err_all["IMD"] = reco_err_all["IMD"].astype(int)
reco_err_all = reco_err_all.merge(pca_err, on="OA21CD", how="left")

print(f"Final DataFrame shape: {reco_err_all.shape}")
if len(reco_err_all) != 188880:
    print(f"Warning: Expected 188880 rows, got {len(reco_err_all)}")

## 3. Compute Per-Group Statistics

In [ ]:
def compute_grouped_stats(df, group_col, n_runs):
    """Compute mean/std/cv across runs and PCA mean for each group."""
    results = []
    for run_idx in range(n_runs):
        grouped = df.groupby(group_col)[f'reco_err_run_{run_idx}'].mean()
        results.append(grouped)

    by_run = pd.DataFrame(results).T
    by_run.columns = [f'run_{i}' for i in range(n_runs)]
    by_run['mean'] = by_run[[f'run_{i}' for i in range(n_runs)]].mean(axis=1)
    by_run['std'] = by_run[[f'run_{i}' for i in range(n_runs)]].std(axis=1)
    by_run['cv'] = by_run['std'] / by_run['mean'] * 100
    by_run['pca'] = df.groupby(group_col)['pca_err'].mean()
    return by_run

imd_by_run = compute_grouped_stats(reco_err_all, 'IMD', n_runs)
density_by_run = compute_grouped_stats(reco_err_all, 'Density_decile', n_runs)
oac_sg_by_run = compute_grouped_stats(reco_err_all, 'Supergroup_codename', n_runs).sort_values('mean')
oac_gr_by_run = compute_grouped_stats(reco_err_all, 'Group', n_runs).sort_values('mean')

print("IMD by run:")
print(imd_by_run[['mean', 'std', 'cv', 'pca']].round(4))
print(f"\nDensity by run:")
print(density_by_run[['mean', 'std', 'cv', 'pca']].round(4))
print(f"\nOAC Supergroup by run:")
print(oac_sg_by_run[['mean', 'std', 'cv', 'pca']].round(4))

## 4. Plotting Function

In [ ]:
COL_AE = '#2b83ba'
COL_PCA = '#d7191c'


def plot_stability_side_by_side(by_run1, by_run2, xlabel1, xlabel2,
                                title_prefix, n_runs,
                                diff_style='bar', marker='s',
                                rotation=0, wrapped_labels=False,
                                col_ae=None, col_pca=None,
                                print_means=False):
    """Side-by-side 2x2 chart (matching 5b style) with error bars from stability runs.

    Top panels:  AE (mean +/- std across runs) vs PCA bars.
    Bottom panels: (AE-PCA)/PCA % with error bars.
    diff_style: 'bar' or 'point_only'.
    """
    from matplotlib.ticker import MaxNLocator

    _col_ae = col_ae or COL_AE
    _col_pca = col_pca or COL_PCA

    fig, axes = plt.subplots(2, 2, figsize=(14, 7),
                             gridspec_kw={'height_ratios': [3, 1], 'wspace': 0.3},
                             sharex='col')

    for col_idx, (by_run, xlabel) in enumerate([
        (by_run1, xlabel1),
        (by_run2, xlabel2),
    ]):
        x = np.arange(len(by_run))
        width = 0.35

        labels = [str(l) for l in by_run.index]
        if wrapped_labels:
            labels = [fill(l, width=21) for l in labels]

        # --- Top panel: AE (mean +/- std) vs PCA ---
        axes[0, col_idx].bar(
            x - width / 2, by_run['mean'], width,
            yerr=by_run['std'], capsize=3,
            label=f'AE (mean \u00b1 std, n={n_runs})',
            color=_col_ae, edgecolor='black', linewidth=0.6)
        axes[0, col_idx].bar(
            x + width / 2, by_run['pca'], width,
            label='PCA',
            color=_col_pca, edgecolor='black', linewidth=0.6)
        axes[0, col_idx].set_xlabel('')
        axes[0, col_idx].set_title(f'{title_prefix} by {xlabel}')
        axes[0, col_idx].tick_params(axis='x', rotation=rotation)
        axes[0, col_idx].legend(title='', frameon=False)
        axes[0, col_idx].spines['top'].set_visible(False)
        axes[0, col_idx].spines['right'].set_visible(False)
        axes[0, col_idx].grid(True, axis='y', linestyle='-', alpha=0.2)
        axes[0, col_idx].set_axisbelow(True)

        # --- Bottom panel: % diff vs PCA ---
        perc_diff = (by_run['mean'] - by_run['pca']) / by_run['pca'] * 100
        perc_diff_std = by_run['std'] / by_run['pca'] * 100
        overall_mean = perc_diff.mean()

        if diff_style == 'point_only':
            axes[1, col_idx].errorbar(x, perc_diff, yerr=perc_diff_std,
                                       fmt='none', color='grey', capsize=3, zorder=1)
            axes[1, col_idx].scatter(x, perc_diff, color='black', s=60, zorder=2,
                                      marker=marker, edgecolors='white', linewidths=0.5)
            axes[1, col_idx].axhline(overall_mean, color='black', linestyle='--',
                                      linewidth=1.0, label=f'Overall: {overall_mean:.1f}%', zorder=0)
            max_dev = max(abs(perc_diff.max() - overall_mean),
                          abs(perc_diff.min() - overall_mean))
            pad = max(max_dev * 0.3, perc_diff_std.max() * 1.2)
            axes[1, col_idx].set_ylim(overall_mean - max_dev - pad,
                                       overall_mean + max_dev + pad)
        else:
            axes[1, col_idx].bar(x, perc_diff, color='#a0a0a0',
                                  edgecolor='black', linewidth=0.6)
            axes[1, col_idx].errorbar(x, perc_diff, yerr=perc_diff_std,
                                       fmt='none', color='black', capsize=3)
            axes[1, col_idx].axhline(0, color='grey', linestyle='-', linewidth=0.8)
            axes[1, col_idx].axhline(overall_mean, color='black', linestyle='--',
                                      linewidth=1.0, label=f'Overall: {overall_mean:.1f}%')

        axes[1, col_idx].set_xticks(x)
        axes[1, col_idx].set_xticklabels(labels, rotation=rotation)
        axes[1, col_idx].set_xlabel(xlabel)
        axes[1, col_idx].set_ylabel('(AE \u2212 PCA) / PCA  [%]')
        axes[1, col_idx].legend(frameon=False, fontsize=7)
        axes[1, col_idx].yaxis.set_major_locator(MaxNLocator(nbins=6))
        axes[1, col_idx].spines['top'].set_visible(False)
        axes[1, col_idx].spines['right'].set_visible(False)
        axes[1, col_idx].grid(True, axis='y', linestyle='-', alpha=0.2)
        axes[1, col_idx].set_axisbelow(True)

        if print_means:
            print(f"\n=== {xlabel} [diff_style={diff_style}] ===")
            summary = pd.DataFrame({
                'AE mean': by_run['mean'],
                'AE std': by_run['std'],
                'PCA': by_run['pca'],
                '% diff': perc_diff,
                '% diff std': perc_diff_std,
            })
            print(summary.round(4))

    axes[0, 0].set_ylabel('Mean Reconstruction Error (RMSE)')
    axes[0, 1].set_ylabel('')
    axes[1, 1].set_ylabel('')

    plt.tight_layout()
    return fig, axes

## 5. Produce Figures (IMD & Density, OAC Supergroup & Group)

In [ ]:
for diff_style in ['bar', 'point_only']:
    print(f"\n{'#'*60}")
    print(f"  IMD / Density  —  {diff_style}")
    print(f"{'#'*60}")
    fig, axes = plot_stability_side_by_side(
        imd_by_run, density_by_run,
        'IMD Decile', 'Density Decile',
        'Reconstruction Error', n_runs,
        diff_style=diff_style, marker='s', rotation=0,
        print_means=True
    )
    plt.show()

    print(f"\n{'#'*60}")
    print(f"  OAC Supergroup / Group  —  {diff_style}")
    print(f"{'#'*60}")
    fig, axes = plot_stability_side_by_side(
        oac_sg_by_run, oac_gr_by_run,
        'OAC SuperGroup', 'OAC Group',
        'Reconstruction Error', n_runs,
        diff_style=diff_style, marker='s', rotation=60,
        wrapped_labels=True,
        print_means=True
    )
    plt.show()

## 6. Rank Stability Analysis

In [ ]:
def compute_rank_stability(by_run_df, n_runs):
    """Compute pairwise rank correlations between runs."""
    correlations = []
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            rho, _ = spearmanr(by_run_df[f'run_{i}'], by_run_df[f'run_{j}'])
            correlations.append(rho)
    return np.mean(correlations), np.std(correlations), correlations

imd_rank_mean, imd_rank_std, _ = compute_rank_stability(imd_by_run, n_runs)
density_rank_mean, density_rank_std, _ = compute_rank_stability(density_by_run, n_runs)
oac_sg_rank_mean, oac_sg_rank_std, _ = compute_rank_stability(oac_sg_by_run, n_runs)
oac_gr_rank_mean, oac_gr_rank_std, _ = compute_rank_stability(oac_gr_by_run, n_runs)

print("Rank Stability (Spearman correlation between runs):")
print(f"  IMD Decile:       {imd_rank_mean:.4f} \u00b1 {imd_rank_std:.4f}")
print(f"  Density Decile:   {density_rank_mean:.4f} \u00b1 {density_rank_std:.4f}")
print(f"  OAC Supergroup:   {oac_sg_rank_mean:.4f} \u00b1 {oac_sg_rank_std:.4f}")
print(f"  OAC Group:        {oac_gr_rank_mean:.4f} \u00b1 {oac_gr_rank_std:.4f}")

In [ ]:
# Spaghetti plots: individual runs + mean +/- std
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, by_run, xlabel, rank_mean in [
    (axes[0], imd_by_run, 'IMD Decile', imd_rank_mean),
    (axes[1], density_by_run, 'Density Decile', density_rank_mean),
    (axes[2], oac_sg_by_run, 'OAC Supergroup', oac_sg_rank_mean),
    (axes[3], oac_gr_by_run, 'OAC Group', oac_gr_rank_mean),
]:
    x = np.arange(len(by_run))
    for i in range(n_runs):
        ax.plot(x, by_run[f'run_{i}'], alpha=0.3, color='seagreen')
    ax.plot(x, by_run['mean'], linewidth=2, color='darkgreen', label='Mean')
    ax.fill_between(x,
                    by_run['mean'].values - by_run['std'].values,
                    by_run['mean'].values + by_run['std'].values,
                    alpha=0.3, color='seagreen')
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('Mean RMSE (%)', fontsize=10)
    ax.set_title(f'{xlabel}\n(Spearman r = {rank_mean:.3f})', fontsize=10, fontweight='bold')
    ax.grid(True, alpha=0.3)

    if xlabel in ['OAC Supergroup', 'OAC Group']:
        ax.set_xticks(x)
        ax.set_xticklabels([s[:8] + '...' if len(s) > 8 else s for s in by_run.index],
                           rotation=45, ha='right', fontsize=7)

plt.suptitle(f'Error Profile Stability Across {n_runs} Runs (500ep lin, {bottleneck}D)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
summary_data = {
    'Grouping': ['IMD Decile', 'Density Decile', 'OAC Supergroup', 'OAC Group'],
    'N Groups': [10, 10, 8, len(oac_gr_by_run)],
    'Mean CV (%)': [imd_by_run['cv'].mean(), density_by_run['cv'].mean(),
                    oac_sg_by_run['cv'].mean(), oac_gr_by_run['cv'].mean()],
    'Max CV (%)': [imd_by_run['cv'].max(), density_by_run['cv'].max(),
                   oac_sg_by_run['cv'].max(), oac_gr_by_run['cv'].max()],
    'Spearman r': [imd_rank_mean, density_rank_mean, oac_sg_rank_mean, oac_gr_rank_mean],
    'Spearman r (std)': [imd_rank_std, density_rank_std, oac_sg_rank_std, oac_gr_rank_std],
}

summary_df = pd.DataFrame(summary_data).round(4)

print("=" * 80)
print("STABILITY SUMMARY: Error by Geography (500 epochs, linear scaling)")
print("=" * 80)

print(f"\nConfiguration:")
print(f"  Bottleneck dimension: {bottleneck}D")
print(f"  Training: 500 epochs, linear scaling")
print(f"  Number of retraining runs: {n_runs}")
print(f"  Number of OAs analyzed: {len(reco_err_all)}")

print(f"\nOverall Reconstruction Error:")
print(f"  AE Mean RMSE: {reco_err_all['ae_mean'].mean():.4f}% \u00b1 {reco_err_all['ae_std'].mean():.4f}%")
print(f"  PCA Mean RMSE: {reco_err_all['pca_err'].mean():.4f}%")
print(f"  Per-OA CV: {reco_err_all['ae_cv'].mean():.2f}%")

print(f"\n{summary_df.to_string(index=False)}")

print(f"\nNotes:")
print(f"- CV = Coefficient of Variation (lower is more stable)")
print(f"- Spearman r = Rank correlation between runs (higher is more stable)")
print(f"- All metrics based on {n_runs} independent retraining runs")
print("=" * 80)